# 07 · ArchTune inside `triton.autotune`

The predictor plugs into Triton through `prune_configs_by`: Triton hands it the full config list,
it returns the top k, and only those are compiled and benchmarked. Here that is timed against the
unpruned autotuner, which compiles and benchmarks all 384 configs.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
DIMS = model.derive_dims(model.load_model_config(cache_dir=RESULTS))
MM_SHAPES = model.matmul_shapes(DIMS)
RMS_SHAPES = model.rmsnorm_shapes(DIMS)
SPLIT = {**model.train_test_split([s.shape_id for s in MM_SHAPES]),
         **model.train_test_split([s.shape_id for s in RMS_SHAPES])}
if MINI:
    small = model.mini_shapes(MM_SHAPES + RMS_SHAPES)
    MM_SHAPES = [s for s in small if s.kernel == "matmul"]
    RMS_SHAPES = [s for s in small if s.kernel == "rmsnorm"]
print(DIMS)
for s in MM_SHAPES + RMS_SHAPES:
    print(f"{s.shape_id:24s} {SPLIT[s.shape_id]:5s} {','.join(s.roles)}")

In [ ]:
import os, pandas as pd
EX = config.EXPERIMENTS
A = gpu.live()
models = run.load_models(RESULTS, ENV["gpu_tag"])
kw = {"reg_model": models[0], "smem_model": models[1]} if models else {}
SHAPES = [s for s in MM_SHAPES if s.shape_id in EX.integration_shapes]
CFGS = model.mini_matmul_space() if MINI else model.matmul_space()
rows = []
for s in SHAPES:
    for k in EX.integration_ks:
        r = model.tune_live(CFGS, s.M, s.N, s.K, prune_by=model.archtune_prune_configs_by(A, k, **kw))
        rows.append({"mode": f"archtune_top{k}", "shape_id": s.shape_id, **r})
        print(rows[-1]["mode"], s.shape_id, f"{r['tuning_s']:.1f}s", f"{r['time_ms_median']:.4f} ms")

Unpruned, on the large shape only: this compiles every config and takes tens of minutes.

In [ ]:
s = next(s for s in SHAPES if s.shape_id == EX.large_shape_id)
r = model.tune_live(CFGS, s.M, s.N, s.K, prune_by=None)
rows.append({"mode": "unpruned", "shape_id": s.shape_id, **r})
print("unpruned", s.shape_id, f"{r['tuning_s']:.1f}s", f"{r['time_ms_median']:.4f} ms")
out = pd.DataFrame(rows); out["best_cfg"] = out["best_cfg"].astype(str)
out.to_parquet(os.path.join(RESULTS, ENV["gpu_tag"], "integration.parquet")); display(out)

## Inductor

For reference, the same matmuls through `torch.compile(mode="max-autotune")`. On GPUs with fewer
than 68 SMs Inductor skips its GEMM template autotuning, so this is its default path there.

In [ ]:
ind = pd.DataFrame([{**model.inductor_compile_and_time(s.M, s.N, s.K), "shape_id": s.shape_id} for s in SHAPES])
ind.to_parquet(os.path.join(RESULTS, ENV["gpu_tag"], "inductor.parquet")); display(ind)